# Market Data Loading

Pull market data from S3 via the Java MarketDataEntry infrastructure using JPype.

**Prerequisites:**
- `gnome-backtest` uber JAR built: `cd gnome-backtest && mvn package -DskipTests`
- AWS credentials configured (`~/.aws/credentials` or env vars)

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

from gnomepy.java import (
    ensure_jvm_started,
    MarketDataClient,
    SchemaType,
)

ensure_jvm_started()

## Configuration

Fill in your exchange, security, schema type, and date range.

In [2]:
# --- Fill these in ---
EXCHANGE_ID = 1
SECURITY_ID = 1
SCHEMA_TYPE = SchemaType.MBP_10
START = datetime(2026, 1, 20, 10, 30)
END = datetime(2026, 1, 20, 12, 0)
BUCKET = "gnome-market-data-prod"
# ---------------------

## Load as Python Schema Objects

In [3]:
client = MarketDataClient(bucket=BUCKET)
schemas = client.load(
    exchange_id=EXCHANGE_ID,
    security_id=SECURITY_ID,
    schema_type=SCHEMA_TYPE,
    start=START,
    end=END,
)
print(f"Loaded {len(schemas)} records")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Loaded 23701 records


In [4]:
# Inspect the first record
if schemas:
    s = schemas[0]
    print(f"Schema type: {s.schema_type}")
    print(f"Event timestamp: {s.event_timestamp}")
    print(f"Full dict: {s.to_dict()}")

Schema type: SchemaType.MBP_10
Event timestamp: 1768905000003000000
Full dict: {'schema_type': 'mbp-10', 'event_timestamp': 1768905000003000000, 'sequence_number': 1768905000003000000, 'exchange_id': 1, 'security_id': 1, 'timestamp_event': 1768905000003000000, 'timestamp_sent': 0, 'timestamp_recv': 1768905000392921206, 'price': 91231000000000, 'size': 3810, 'side': 'Bid', 'action': 'Trade', 'depth': 255, 'sequence': 1768905000003000000, 'bid_price_0': 91219000000000, 'ask_price_0': 91220000000000, 'bid_size_0': 13757570, 'ask_size_0': 712550, 'bid_count_0': 41, 'ask_count_0': 2, 'bid_price_1': 91218000000000, 'ask_price_1': 91222000000000, 'bid_size_1': 1411280, 'ask_size_1': 3130, 'bid_count_1': 23, 'ask_count_1': 2, 'bid_price_2': 91217000000000, 'ask_price_2': 91223000000000, 'bid_size_2': 377390, 'ask_size_2': 1151040, 'bid_count_2': 3, 'ask_count_2': 4, 'bid_price_3': 91216000000000, 'ask_price_3': 91224000000000, 'bid_size_3': 674380, 'ask_size_3': 240, 'bid_count_3': 4, 'ask_cou

## Load as DataFrame

In [5]:
df = client.load_dataframe(
    exchange_id=EXCHANGE_ID,
    security_id=SECURITY_ID,
    schema_type=SCHEMA_TYPE,
    start=START,
    end=END,
)
print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (23701, 74)


,schema_type,event_timestamp,sequence_number,exchange_id,security_id,timestamp_event,timestamp_sent,timestamp_recv,price,size,...,bid_size_8,ask_size_8,bid_count_8,ask_count_8,bid_price_9,ask_price_9,bid_size_9,ask_size_9,bid_count_9,ask_count_9
0,mbp-10,1768905000003000000,1768905000003000000,1,1,1768905000003000000,0,1768905000392921206,91231000000000,3810,...,1070520,3130,5,2,91210000000000,91230000000000,1801120,1172580,9,5
1,mbp-10,1768905000322000000,1768905000322000000,1,1,1768905000322000000,0,1768905000581963094,91231000000000,3810,...,1102500,11950,5,4,91220000000000,91239000000000,605560,1676790,4,6
2,mbp-10,1768905000910000000,1768905000910000000,1,1,1768905000910000000,0,1768905001281732984,91231000000000,3810,...,2850700,11950,7,4,91219000000000,91239000000000,798620,225930,5,4
3,mbp-10,1768905001003000000,1768905001003000000,1,1,1768905001003000000,0,1768905001381733687,91230000000000,120,...,2850700,11950,7,4,91219000000000,91239000000000,798620,225930,5,4
4,mbp-10,1768905001334000000,1768905001334000000,1,1,1768905001334000000,0,1768905001581797471,91229000000000,1100,...,2850700,11950,7,4,91219000000000,91239000000000,798620,225930,5,4


## Available Schema Types

```
SchemaType.MBO       - Market by Order
SchemaType.MBP_10    - Market by Price (10 levels)
SchemaType.MBP_1     - Market by Price (1 level)
SchemaType.BBO_1S    - Best Bid/Offer (1s)
SchemaType.BBO_1M    - Best Bid/Offer (1m)
SchemaType.TRADES    - Trade messages
SchemaType.OHLCV_1S  - OHLCV (1s bars)
SchemaType.OHLCV_1M  - OHLCV (1m bars)
SchemaType.OHLCV_1H  - OHLCV (1h bars)
```

## Load Multiple Schema Types

In [6]:
# Example: load trades alongside MBP data
trades_df = client.load_dataframe(
    exchange_id=EXCHANGE_ID,
    security_id=SECURITY_ID,
    schema_type=SchemaType.TRADES,
    start=START,
    end=END,
)
print(f"Trades: {len(trades_df)} records")
trades_df.head()

Trades: 13891 records


,schema_type,event_timestamp,sequence_number,exchange_id,security_id,timestamp_event,timestamp_sent,timestamp_recv,price,size,side,action,sequence,depth
0,trades,1768905000003000000,1768905000003000000,1,1,1768905000003000000,0,1768905000392921206,91231000000000,3810,Bid,Trade,1768905000003000000,255
1,trades,1768905001003000000,1768905001003000000,1,1,1768905001003000000,0,1768905001381733687,91230000000000,120,Bid,Trade,1768905001003000000,255
2,trades,1768905001334000000,1768905001334000000,1,1,1768905001334000000,0,1768905001581797471,91229000000000,1100,Ask,Trade,1768905001334000000,255
3,trades,1768905002042000000,1768905002042000000,1,1,1768905002042000000,0,1768905002323919061,91229000000000,11660,Ask,Trade,1768905002042000000,255
4,trades,1768905002042000000,1768905002042000000,1,1,1768905002042000000,0,1768905002323919061,91229000000000,200,Ask,Trade,1768905002042000000,255


## Price Scaling

Raw prices are scaled integers (x 1,000,000,000). Divide to get human-readable values.

In [7]:
PRICE_SCALE = 1_000_000_000

if not df.empty and "bid_price_0" in df.columns:
    df["mid_price"] = (df["bid_price_0"] + df["ask_price_0"]) / (2 * PRICE_SCALE)
    df["spread"] = (df["ask_price_0"] - df["bid_price_0"]) / PRICE_SCALE
    print(df[["mid_price", "spread"]].describe())

          mid_price        spread
count  23701.000000  23701.000000
mean   91247.748428      1.082613
std       68.321501      0.716539
min    91091.500000      1.000000
25%    91194.500000      1.000000
50%    91261.500000      1.000000
75%    91306.500000      1.000000
max    91375.500000     20.000000


## Quick Plot

In [ ]:
import matplotlib.pyplot as plt

if not df.empty and "mid_price" in df.columns:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax1.plot(df["mid_price"], linewidth=0.5)
    ax1.set_ylabel("Mid Price")
    ax1.set_title(f"Exchange {EXCHANGE_ID} / Security {SECURITY_ID}")

    ax2.plot(df["spread"], linewidth=0.5, color="orange")
    ax2.set_ylabel("Spread")
    ax2.set_xlabel("Record Index")

    plt.tight_layout()
    plt.show()